In [6]:
from generate import generate
import csv
from pathlib import Path
import numpy as np
from scipy.ndimage import gaussian_filter
from PIL import Image
import os
import matplotlib.pyplot as plt
import cv2 as cv
import sys
import pandas as pd
from pandas.errors import EmptyDataError

In [40]:
# Generation of images function
def generate_img(nr_of_img, nr_tracks):
    
    output_img = Path(r'C:\Users\User\Desktop\RADON_images')
    output_csv = Path(r'C:\Users\User\Desktop\RADON_tables')
    
    for item in output_img.iterdir():
        if item.is_file():
            item.unlink()
    for item in output_csv.iterdir():
        if item.is_file():
            item.unlink()
            
    for i in range(0,nr_of_img): 
        if __name__ == '__main__':
            F_output = output_img / f'synthetic_radon_{i}.png'
            F_truth = output_csv / f'synthetic_radon_truth_{i}.csv'
            
            IMG_pixel = (800,1000)
            n_tracks = nr_tracks
            seed = i
    
            generate(output=F_output, truth=F_truth, shape=IMG_pixel,
                 n_tracks=n_tracks, seed=seed)
    
    image_count = sum(1 for item in output_img.iterdir() if item.is_file())
    if image_count != (i+1):
        print("Error,",image_count,"files when there should be", i+1)

In [41]:
# Detection of circles in generated images

def detection(nr_img_detect):
    det_csv = Path(r'C:\Users\User\Desktop\RADON_detected')
    for item in det_csv.iterdir():
        if item.is_file():
            item.unlink()
    
    for j in range (0,nr_img_detect):
        specs = []
        img = cv.imread(fr'C:\Users\User\Desktop\RADON_images\synthetic_radon_{j}.png', cv.IMREAD_COLOR)
        if img is None:
            print ('Error opening image!')
        
        gray = cv.cvtColor(img, cv.COLOR_BGR2GRAY)
        gray=cv.blur(gray,(4,4))

        rows = gray.shape[0]
        circles = cv.HoughCircles(gray, cv.HOUGH_GRADIENT, 1, minDist =7 , param1 = 10, param2=30,  minRadius=7, maxRadius=31)
        nr = 0
        if circles is not None: 
            circles= np.uint16(np.around(circles))
            for i in circles[0,:]:
                center = (i[0],i[1])
                cv.circle(img, center, 1, (0, 100, 100), 3)
                radius = i[2]
                cv.circle(img, center, radius, (255, 0, 255), 3)
                specs.append((center[0],center[1],radius))
                with open(fr"C:\Users\User\Desktop\RADON_detected\data{j}.csv", "w", newline="") as file:
                    writer = csv.writer(file)
                    writer.writerow(['x_px','y_px','radius_px'])
                    writer.writerows(specs)   
                nr =nr+1  
           
            print ('data',j,'   detected:',nr)
        else: 
            with open(fr"C:\Users\User\Desktop\RADON_detected\data{j}.csv", "w", newline="") as file: 
                if specs: 
                    writer = csv.writer(file)
                    writer.writerow(['x_px','y_px','radius_px'])
                    writer.writerows(specs)         
            print ('data',j,'   detected:',nr)
            
    det_count = sum(1 for item in det_csv.iterdir() if item.is_file())
    if det_count != (j+1):
        print("Error,",det_count,"files when there should be", j+1)

In [48]:
x_diff : int
y_diff : int 
r_diff : int
def compare(nr_data_compare): 
    global x_diff, y_diff, r_diff
    for i in range (0,nr_data_compare):
    
        real_data_df= pd.read_csv(fr"C:\Users\User\Desktop\RADON_tables\synthetic_radon_truth_{i}.csv")
        real_data_df= real_data_df[['x_px','y_px','radius_px']]
        rd = real_data_df.to_numpy()
        rd = np.round(rd).astype(int)
        rd = rd[np.lexsort((rd[:, 1], rd[:, 0]))]


        try: 
            det_data_df= pd.read_csv(fr"C:\Users\User\Desktop\RADON_detected\data{i}.csv")
            dd = det_data_df.to_numpy()
            dd = np.round(dd).astype(int)
            dd = dd[np.lexsort((dd[:, 1], dd[:, 0]))]
        except EmptyDataError: 
            dd = np.array([])

        #to remove copies in dd, however we run the risk of removing two circles that are just very close to each other
        kept = []
        for row in dd:
            if not any(abs(r[0] - row[0]) <= 5 and abs(r[1] - row[1]) <= 5 for r in kept):
                kept.append(row)

        dd_filtered = np.array(kept, dtype=dd.dtype).reshape(-1, 3)
        dd=dd_filtered

        print(len(dd))

        
        id_real=0
        id_det=0
        found_id_det = np.array([])
        found_id_real = np.array([])
        
        x_diff=abs(rd[id_real,0] - dd[id_det,0])
        y_diff=abs(rd[id_real,1] - dd[id_det,1])
        r_diff=abs(rd[id_real,2] - dd[id_det,2])
        
        if dd.size == 0 : 
            print(f"In image {i}: NO CIRCLES DETECTED")
        else: 
            while id_det < len(dd):
                if id_real >= len(rd): 
                    id_real=0  
                    id_det=id_det+1
                elif x_diff <= 10 and y_diff <=10:
                    found_id_det = np.append(found_id_det, int(id_det))
                    found_id_real = np.append(found_id_real, int(id_real))
                    id_det=id_det+1
                    id_real=0
                else:
                    id_real=id_real + 1

            real_unfound = np.delete(dd, found_id_det.astype(int), axis=0)
            det_wrong = np.delete(rd, found_id_real.astype(int), axis=0)

            



            print(f"In image {i}:   ", "Extras in detection: ",len(real_unfound),"  Unfound: ", len(det_wrong))


In [49]:
#generate_img(nr_of_img=50, nr_tracks=50)
detection(nr_img_detect=50)
compare(nr_data_compare=50)



data 0    detected: 47
data 1    detected: 48
data 2    detected: 48
data 3    detected: 50
data 4    detected: 47
data 5    detected: 49
data 6    detected: 48
data 7    detected: 50
data 8    detected: 50
data 9    detected: 50
data 10    detected: 49
data 11    detected: 50
data 12    detected: 50
data 13    detected: 49
data 14    detected: 49
data 15    detected: 51
data 16    detected: 49
data 17    detected: 46
data 18    detected: 51
data 19    detected: 50
data 20    detected: 50
data 21    detected: 51
data 22    detected: 49
data 23    detected: 48
data 24    detected: 47
data 25    detected: 50
data 26    detected: 50
data 27    detected: 49
data 28    detected: 50
data 29    detected: 49
data 30    detected: 48
data 31    detected: 48
data 32    detected: 49
data 33    detected: 47
data 34    detected: 49
data 35    detected: 49
data 36    detected: 49
data 37    detected: 47
data 38    detected: 48
data 39    detected: 50
data 40    detected: 49
data 41    detected: 48
da